In [2]:
# Cell 1: Imports
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import metrics


In [3]:

# Cell 2: Load Dataset
housing = pd.read_csv(r"C:\Users\VICTUS\Downloads\housing.csv")
housing.head()


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41,880,129.0,322,126,8.3252,452600,NEAR BAY
1,-122.22,37.86,21,7099,1106.0,2401,1138,8.3014,358500,NEAR BAY
2,-122.24,37.85,52,1467,190.0,496,177,7.2574,352100,NEAR BAY
3,-122.25,37.85,52,1274,235.0,558,219,5.6431,341300,NEAR BAY
4,-122.25,37.85,52,1627,280.0,565,259,3.8462,342200,NEAR BAY


In [4]:
# Cell 3: Outlier Removal Function
def findOutliers(df, col):
    x = df[col].describe()
    Q25 = x['25%']
    Q75 = x['75%']
    IQR = Q75 - Q25
    lowerBound = Q25 - 1.5 * IQR
    upperBound = Q75 + 1.5 * IQR
    data = df[(df[col] < lowerBound) | (df[col] > upperBound)]
    print(f"No. of outliers: {len(data[col])}, out of {len(df[col])}")
    z = df[~df[col].isin(data[col])]
    return z

# Clean outliers in total_rooms
housing = findOutliers(housing, "total_rooms")


No. of outliers: 1287, out of 20640


In [5]:

# Cell 4: Handle Missing Values
imputer = SimpleImputer(missing_values=np.nan, strategy="median")
housing.iloc[:, 4:5] = imputer.fit_transform(housing.iloc[:, 4:5])

# Encode Categorical Feature
labelEncoder = LabelEncoder()
housing["ocean_proximity"] = labelEncoder.fit_transform(housing["ocean_proximity"])


In [6]:

# Cell 5: Visualizing Geographic & Price Distribution (Plotly Map)
fig_map = px.scatter_mapbox(
    housing.sample(n=3000, random_state=42),  # Subsampled for responsive rendering
    lat="latitude",
    lon="longitude",
    color="median_house_value",
    size="population",
    color_continuous_scale=px.colors.cyclical.IceFire,
    size_max=15,
    zoom=5,
    mapbox_style="carto-positron",
    title="California Housing Map: House Values & Population Density"
)
fig_map.show()


C:\Users\VICTUS\AppData\Local\Temp\ipykernel_35416\1031291627.py:2: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_map = px.scatter_mapbox(


In [7]:

# Cell 6: Correlation Heatmap (Plotly Interactive Heatmap)
corr_matrix = housing.corr().round(2)
fig_corr = px.imshow(
    corr_matrix,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="Viridis",
    title="Correlation Heatmap of Housing Features"
)
fig_corr.show()



In [8]:
# Cell 7: Model Training (Linear Regression)
housing_inp = housing.drop("median_house_value", axis=1)
housing_out = housing["median_house_value"]

X_train, X_test, y_train, y_test = train_test_split(
    housing_inp, housing_out, test_size=0.4, random_state=101
)

independent_scaler = StandardScaler()
X_train = independent_scaler.fit_transform(X_train)
X_test = independent_scaler.transform(X_test)

lm = LinearRegression()
lm.fit(X_train, y_train)

predictions = lm.predict(X_test)

print(f"Intercept: {lm.intercept_:.2f}")
print(f"MAE:  {metrics.mean_absolute_error(y_test, predictions):.2f}")
print(f"MSE:  {metrics.mean_squared_error(y_test, predictions):.2f}")
print(f"RMSE: {np.sqrt(metrics.mean_squared_error(y_test, predictions)):.2f}")

# Cell 8: Residual Distribution Plot
residuals = y_test - predictions
fig_hist = px.histogram(
    residuals, 
    nbins=50,
    title="Residuals (Actual - Predicted) Distribution",
    labels={'value': 'Residual Amount'},
    marginal="rug"
)
fig_hist.show()


Intercept: 205708.43
MAE:  50752.98
MSE:  4749633215.96
RMSE: 68917.58


In [9]:

# Cell 9: Actual vs Predicted Housing Prices Line Plot
df_final = pd.DataFrame({'Actual': y_test, 'Predicted': predictions}).reset_index(drop=True)

fig_compare = go.Figure()
fig_compare.add_trace(go.Scatter(y=df_final['Actual'][:50], mode='lines+markers', name='Actual'))
fig_compare.add_trace(go.Scatter(y=df_final['Predicted'][:50], mode='lines+markers', name='Predicted'))

fig_compare.update_layout(
    title="First 50 Samples: Actual vs Predicted Median House Value",
    xaxis_title="Sample Index",
    yaxis_title="House Value ($)",
    hovermode="x unified"
)
fig_compare.show()


In [10]:

# Cell 10: Regression Scatter Plot (Actual vs Predicted Joint Plot equivalent)
fig_reg = px.scatter(
    df_final, 
    x='Actual', 
    y='Predicted', 
    trendline="ols",
    trendline_color_override="red",
    opacity=0.4,
    title="Actual vs Predicted House Values with OLS Trendline"
)
fig_reg.show()